In [12]:
import json
import time
from pathlib import Path

import numpy as np
import pandas as pd
import requests

In [13]:
input_csv = Path(
    "/Users/andrewweiland/Chopra_Lab_Research/"
    "moo_selection/curated_200_approved_drug_molecules.csv"
)
compounds = pd.read_csv(input_csv)
if "smiles" not in compounds.columns:
    raise ValueError("The input CSV must contain a 'smiles' column.")
print(f"Loaded {len(compounds)} compounds")
compounds.head()

Loaded 200 compounds


,candidate_id,chembl_id,preferred_name,smiles,standard_inchi_key,molecular_formula,molecular_weight,alogp,polar_surface_area,hbond_acceptors,...,first_approval_year,max_phase,oral,parenteral,topical,prodrug,black_box_warning,source_database,source_record_url,source_retrieved_date
0,MOO001,CHEMBL2205807,ABAMETAPIR,Cc1ccc(-c2ccc(C)cn2)nc1,PTRATZCAGVBFIQ-UHFFFAOYSA-N,C12H12N2,184.24,2.76,25.78,2,...,2020.0,4,False,False,True,False,False,ChEMBL,https://www.ebi.ac.uk/chembl/explore/compound/...,2026-08-29
1,MOO002,CHEMBL271227,ABIRATERONE ACETATE,CC(=O)O[C@H]1CC[C@@]2(C)C(=CC[C@@H]3[C@@H]2CC[...,UVIQSJCZCSLXRZ-UBUQANBQSA-N,C26H33NO2,391.56,5.97,39.19,3,...,2011.0,4,True,False,False,True,False,ChEMBL,https://www.ebi.ac.uk/chembl/explore/compound/...,2026-08-29
2,MOO003,CHEMBL1201327,ACETRIZOIC ACID,CC(=O)Nc1c(I)cc(I)c(C(=O)O)c1I,GNOGSFBXBWBTIG-UHFFFAOYSA-N,C9H6I3NO3,556.86,3.16,66.40,2,...,1982.0,4,False,False,True,False,False,ChEMBL,https://www.ebi.ac.uk/chembl/explore/compound/...,2026-08-29
3,MOO004,CHEMBL3940890,ACORAMIDIS,Cc1n[nH]c(C)c1CCCOc1cc(C(=O)O)ccc1F,WBFUHHBPNXWNCC-UHFFFAOYSA-N,C15H17FN2O3,292.31,2.88,75.21,3,...,2024.0,4,True,False,False,False,False,ChEMBL,https://www.ebi.ac.uk/chembl/explore/compound/...,2026-08-29
4,MOO005,CHEMBL634,ALFENTANIL,CCC(=O)N(c1ccccc1)C1(COC)CCN(CCn2nnn(CC)c2=O)CC1,IDBPHNDTYPBSNI-UHFFFAOYSA-N,C21H32N6O3,416.53,1.38,85.49,8,...,1986.0,4,False,True,False,False,True,ChEMBL,https://www.ebi.ac.uk/chembl/explore/compound/...,2026-08-29


In [137]:
# Change these two values for the microservice you want to test.
microservice_port = 18020
property_name = "hERG"

microservice_url = f"http://localhost:{microservice_port}"
pause_seconds = 3.0
connect_timeout_seconds = 10
read_timeout_seconds = 60
max_consecutive_failures = 3

print(f"Testing {property_name!r} at {microservice_url}")

Testing 'hERG' at http://localhost:18020


In [132]:
def check_health(timeout=10):
    response = requests.get(f"{microservice_url}/health", timeout=timeout)
    response.raise_for_status()
    return response.json()

check_health()

{'status': 'healthy', 'message': 'Admetica microservice is running'}

In [135]:
test_smiles = str(compounds.iloc[0]["smiles"])
test_response = requests.post(
    f"{microservice_url}/smi",
    json={"smiles": test_smiles, "property": [property_name]},
    timeout=(connect_timeout_seconds, read_timeout_seconds),
)
print(test_response.status_code)
test_response.json()

200


{'smiles': 'Cc1ccc(-c2ccc(C)cn2)nc1',
 'status': 'success',
 'results': {'hERG': {'property': 'hERG',
   'status': 'success',
   'results': 0.0,
   'error': None}},
 'error': None}

In [125]:
def extract_prediction(body, expected_property):
    top_status = str(
        body.get("status", "")
    ).lower()

    if top_status != "success":
        raise RuntimeError(
            f"Microservice returned an error: {body}"
        )

    results = body.get("results", {})
    result = results.get(expected_property)

    if result is None:
        available = list(results)

        raise KeyError(
            f"Property {expected_property!r} was not returned; "
            f"available: {available}"
        )

    property_status = str(
        result.get("status", "")
    ).lower()

    if property_status != "success":
        error_message = result.get(
            "error",
            "Unknown property prediction error",
        )

        raise RuntimeError(
            f"Prediction of {expected_property!r} failed: "
            f"{error_message}"
        )

    value = result.get("results")

    if value is None:
        value = result.get("prediction")

    if value is None:
        raise ValueError(
            f"Property {expected_property!r} reported success "
            "but returned no numeric value."
        )

    return float(value)

In [126]:
session = requests.Session()

def query_microservice(smiles):
    started = time.perf_counter()
    response = session.post(
        f"{microservice_url}/smi",
        json={"smiles": smiles, "property": [property_name]},
        timeout=(connect_timeout_seconds, read_timeout_seconds),
    )
    elapsed = time.perf_counter() - started
    response.raise_for_status()
    body = response.json()
    prediction = extract_prediction(body, property_name)
    return prediction, elapsed, body

In [127]:
test_prediction, test_elapsed, test_body = query_microservice(test_smiles)
print(f"Prediction: {test_prediction}")
print(f"Response time: {test_elapsed:.3f} seconds")
test_body

Prediction: 0.4062
Response time: 0.074 seconds


{'smiles': 'Cc1ccc(-c2ccc(C)cn2)nc1',
 'status': 'success',
 'results': {'AMES': {'property': 'AMES',
   'status': 'success',
   'results': 0.4062,
   'error': None}},
 'error': None}

In [128]:
response = requests.post(
    f"{microservice_url}/smi",
    json={
        "smiles": test_smiles,
        "property": [property_name],
    },
    timeout=(10, 30),
)

print("HTTP status:", response.status_code)

test_body = response.json()

print(json.dumps(test_body, indent=2))

HTTP status: 200
{
  "smiles": "Cc1ccc(-c2ccc(C)cn2)nc1",
  "status": "success",
  "results": {
    "AMES": {
      "property": "AMES",
      "status": "success",
      "results": 0.4062,
      "error": null
    }
  },
  "error": null
}


In [138]:
request_rows = []
consecutive_failures = 0

for position, (_, compound) in enumerate(compounds.iterrows(), start=1):
    smiles = str(compound["smiles"])
    print(f"{position}/{len(compounds)}: {smiles}", flush=True)
    started = time.perf_counter()
    try:
        prediction, elapsed, body = query_microservice(smiles)
        request_rows.append({
            "request_number": position,
            "smiles": smiles,
            "property": property_name,
            "prediction": prediction,
            "response_seconds": elapsed,
            "status": "success",
            "error": None,
            "raw_response": json.dumps(body),
        })
        consecutive_failures = 0
        print(f"  success in {elapsed:.3f}s", flush=True)
    except Exception as exc:
        elapsed = time.perf_counter() - started
        consecutive_failures += 1
        request_rows.append({
            "request_number": position,
            "smiles": smiles,
            "property": property_name,
            "prediction": np.nan,
            "response_seconds": elapsed,
            "status": "error",
            "error": str(exc),
            "raw_response": None,
        })
        print(f"  ERROR after {elapsed:.3f}s: {exc}", flush=True)
        if consecutive_failures >= max_consecutive_failures:
            print("Stopping after repeated failures.", flush=True)
            break
    time.sleep(pause_seconds)

1/200: Cc1ccc(-c2ccc(C)cn2)nc1
  success in 0.172s
2/200: CC(=O)O[C@H]1CC[C@@]2(C)C(=CC[C@@H]3[C@@H]2CC[C@]2(C)C(c4cccnc4)=CC[C@@H]32)C1
  success in 0.322s
3/200: CC(=O)Nc1c(I)cc(I)c(C(=O)O)c1I
  ERROR after 0.004s: HTTPConnectionPool(host='localhost', port=18020): Max retries exceeded with url: /smi (Caused by NewConnectionError("HTTPConnection(host='localhost', port=18020): Failed to establish a new connection: [Errno 61] Connection refused"))
4/200: Cc1n[nH]c(C)c1CCCOc1cc(C(=O)O)ccc1F
  ERROR after 0.004s: HTTPConnectionPool(host='localhost', port=18020): Max retries exceeded with url: /smi (Caused by NewConnectionError("HTTPConnection(host='localhost', port=18020): Failed to establish a new connection: [Errno 61] Connection refused"))
5/200: CCC(=O)N(c1ccccc1)C1(COC)CCN(CCn2nnn(CC)c2=O)CC1
  ERROR after 0.004s: HTTPConnectionPool(host='localhost', port=18020): Max retries exceeded with url: /smi (Caused by NewConnectionError("HTTPConnection(host='localhost', port=18020): Failed to

In [11]:
request_results = pd.DataFrame(request_rows)
output_csv = Path(
    "/Users/andrewweiland/Chopra_Lab_Research/"
    "moo_selection/docker_requests_test_results.csv"
)
request_results.to_csv(output_csv, index=False)
print(request_results["status"].value_counts(dropna=False))
print(f"Median response time: {request_results['response_seconds'].median():.3f}s")
print(f"Maximum response time: {request_results['response_seconds'].max():.3f}s")
print(f"Saved: {output_csv}")
request_results.tail(10)

status
success    200
Name: count, dtype: int64
Median response time: 0.056s
Maximum response time: 0.101s
Saved: /Users/andrewweiland/Chopra_Lab_Research/moo_selection/docker_requests_test_results.csv


,request_number,smiles,property,prediction,response_seconds,status,error,raw_response
190,191,CC1(C)C[C@@H]2CCCNc3cccc(n3)S(=O)(=O)NC(=O)c3c...,Solubility_AqSolDB,-5.515146,0.066573,success,None,"{""smiles"": ""CC1(C)C[C@@H]2CCCNc3cccc(n3)S(=O)(..."
191,192,CCCc1nc(C)c2c(=O)nc(-c3cc(S(=O)(=O)N4CCN(CC)CC...,Solubility_AqSolDB,-3.978244,0.061579,success,None,"{""smiles"": ""CCCc1nc(C)c2c(=O)nc(-c3cc(S(=O)(=O..."
192,193,COc1ccc(CCN(C)CCCC(C#N)(c2ccc(OC)c(OC)c2)C(C)C...,Solubility_AqSolDB,-4.395088,0.060396,success,None,"{""smiles"": ""COc1ccc(CCN(C)CCCC(C#N)(c2ccc(OC)c..."
193,194,COC(=O)Nc1c(N)nc(-c2nn(Cc3ccccc3F)c3ncc(F)cc23...,Solubility_AqSolDB,-5.011019,0.060042,success,None,"{""smiles"": ""COC(=O)Nc1c(N)nc(-c2nn(Cc3ccccc3F)..."
194,195,Nc1ncnc2c1ncn2[C@@H]1O[C@H](CO)[C@@H](O)[C@@H]1O,Solubility_AqSolDB,-1.325562,0.049780,success,None,"{""smiles"": ""Nc1ncnc2c1ncn2[C@@H]1O[C@H](CO)[C@..."
195,196,N#C[C@@H]1CCCN1C(=O)CNC12CC3CC(CC(O)(C3)C1)C2,Solubility_AqSolDB,-1.943309,0.053117,success,None,"{""smiles"": ""N#C[C@@H]1CCCN1C(=O)CNC12CC3CC(CC(..."
196,197,CCOc1ccccc1OCC1CNCCO1,Solubility_AqSolDB,-1.004708,0.047758,success,None,"{""smiles"": ""CCOc1ccccc1OCC1CNCCO1"", ""status"": ..."
197,198,C[C@@H](Nc1nc(N[C@H](C)C(F)(F)F)nc(-c2cccc(Cl)...,Solubility_AqSolDB,-5.292754,0.060296,success,None,"{""smiles"": ""C[C@@H](Nc1nc(N[C@H](C)C(F)(F)F)nc..."
198,199,CN(CCO)CC(O)Cn1cnc2c1c(=O)n(C)c(=O)n2C,Solubility_AqSolDB,0.059256,0.053190,success,None,"{""smiles"": ""CN(CCO)CC(O)Cn1cnc2c1c(=O)n(C)c(=O..."
199,200,O=C[C@H](O)[C@@H](O)[C@H](O)CO,Solubility_AqSolDB,0.332233,0.043005,success,None,"{""smiles"": ""O=C[C@H](O)[C@@H](O)[C@H](O)CO"", ""..."
